# 9.1 订单簿微观结构：LOB、价差与订单流

## 为什么要研究订单簿？

传统量化分析看**分钟级或日级**数据，使用价格和成交量。
高频交易（HFT）在**毫秒级**就完成一笔交易，它关注的是：

- 下一秒价格是涨 0.01 元还是跌 0.01 元？
- 谁在买？谁在卖？订单流是否存在不平衡？
- 如何在 1 毫秒内完成最优报价，同时控制库存风险？

这些问题都依赖于对**订单簿微观结构**的深入理解。

## 微观结构的核心问题

> 价格是如何从一个水平移动到另一个水平的？
> 每一笔成交都是买方和卖方之间的博弈——
> 谁先让步，谁付出成本，谁获取价差？

## 学习目标
- 理解限价委托订单簿（LOB）的结构和运行机制
- 掌握买卖价差的来源和意义
- 理解订单流不平衡（OFI）对价格的短期预测
- 分析 Kyle's Lambda（价格冲击系数）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)
print('OK')


In [ ]:
import matplotlib.pyplot as plt

# 1. 设置系统自带的中文字体（这里使用黑体 SimHei）
plt.rcParams['font.sans-serif'] = ['SimHei']  # 如果你想用微软雅黑，可以改成 ['Microsoft YaHei']

# 2. 解决更换字体后，负号（-）显示为方块的问题
plt.rcParams['axes.unicode_minus'] = False

## 第一部分：限价委托订单簿（LOB）的结构

### 什么是订单簿？

所有处于「等待成交」状态的限价订单，按价格排列形成两列：

```
卖方（Ask Side）            买方（Bid Side）
价格    数量                 价格    数量
100.06  1200  ←         →   100.00  800
100.04   800  ←  最优卖价  →   99.98  1500   ← 最优买价
100.02   500  ←  Spread  →   99.96  2000
            ↑ Best Ask  Best Bid ↑
            Spread = 100.02 - 100.00 = 0.02
```

### 关键术语

| 术语 | 公式 | 含义 |
|------|------|------|
| **Best Bid** | 买方最高报价 | 市价卖单可立即成交的价格 |
| **Best Ask** | 卖方最低报价 | 市价买单可立即成交的价格 |
| **中间价（Mid）** | (Bid+Ask)/2 | 市场公认的「公允价格」 |
| **Spread** | Ask - Bid | 流动性成本，也是做市商收入来源 |
| **深度（Depth）** | 各价位的委托数量 | 市场承接能力 |

### 为什么价差（Spread）不为零？

做市商提供流动性（随时愿意买卖），承担库存风险，因此要求补偿——
价差就是这个补偿。市场流动性越好，价差越小（竞争越激烈）。


In [ ]:
# 可视化完整订单簿深度
bid_prices = [100.00, 99.98, 99.96, 99.94, 99.92]
ask_prices = [100.02, 100.04, 100.06, 100.08, 100.10]
bid_vols   = [800, 1500, 2000, 3000, 1200]
ask_vols   = [500, 800, 1200, 600, 900]
mid = (bid_prices[0] + ask_prices[0]) / 2
spread = ask_prices[0] - bid_prices[0]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(bid_prices, [-v for v in bid_vols], color='#2ecc71', alpha=0.75, label='买方委托（Bid）')
ax.barh(ask_prices, ask_vols, color='#e74c3c', alpha=0.75, label='卖方委托（Ask）')
ax.axhline(mid, color='blue', lw=2, linestyle='--',
           label=f'中间价 Mid={mid:.2f}')
ax.axhspan(bid_prices[0], ask_prices[0], alpha=0.15, color='yellow',
           label=f'价差 Spread={spread:.2f} ({spread/mid*10000:.1f}bp)')
ax.set_xlim(-4000, 2000)
ax.set_xlabel('委托数量（负数=买方，正数=卖方）')
ax.set_ylabel('价格')
ax.set_title('限价委托订单簿（LOB）深度可视化
绿色=买方委托，红色=卖方委托')
ax.legend(loc='right', fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f'Best Bid: {bid_prices[0]}  Best Ask: {ask_prices[0]}')
print(f'中间价: {mid}  价差: {spread} ({spread/mid*10000:.1f}bps)')
print(f'买方前5档深度: {sum(bid_vols)} 股  卖方前5档深度: {sum(ask_vols)} 股')


## 第二部分：订单流不平衡（OFI）——短期价格预测因子

### 什么是主动成交？

每笔成交都由一方「主动」发起：
- **买方主动（Buyer-initiated）**：机构用市价单买入，价格通常上升压力
- **卖方主动（Seller-initiated）**：机构用市价单卖出，价格通常下行压力

### 订单流不平衡（Order Flow Imbalance, OFI）

$$\text{OFI} = \sum_{\tau=1}^{t} (V_{\tau}^{\text{buy}} - V_{\tau}^{\text{sell}})$$

OFI > 0 → 买压强，预测短期价格上涨  
OFI < 0 → 卖压强，预测短期价格下跌

### OFI 的预测原理

大机构在买入时，为了避免推高价格，会把订单拆成大量小单分批执行。
在成交过程中，OFI 逐渐积累，价格慢慢上移——这就是**订单流的信息含量**。


In [ ]:
np.random.seed(42)
n_trades = 2000
# 随机生成买卖方向（主动买=1，主动卖=-1）
# 微弱买压：p=0.52
direction = np.random.choice([1, -1], n_trades, p=[0.52, 0.48])
volume    = np.random.randint(100, 2000, n_trades)
mid_start = 100.0

# 价格冲击：主动买单推价格上涨，卖单压价格
price_impact  = 0.0001 * volume * direction
mid_prices    = mid_start + np.cumsum(price_impact)

# OFI: 滚动20笔净订单流
signed_vol = direction * volume
ofi        = pd.Series(signed_vol).rolling(20).sum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(mid_prices, color='steelblue', lw=1, alpha=0.9)
ax1.set_title('模拟中间价走势（受订单流推动）')
ax1.set_ylabel('价格'); ax1.grid(alpha=0.3)

ax2.plot(ofi, color='darkorange', lw=1)
ax2.axhline(0, color='black', lw=0.8)
ax2.fill_between(range(n_trades), ofi, 0, where=(ofi>0), alpha=0.3, color='green')
ax2.fill_between(range(n_trades), ofi, 0, where=(ofi<0), alpha=0.3, color='red')
ax2.set_title('订单流不平衡（OFI，20笔滚动）
绿=买压，红=卖压')
ax2.set_ylabel('OFI'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# OFI 对短期价格变化的预测力
ofi_lag = ofi.shift(1).dropna()
price_chg = pd.Series(mid_prices).diff().dropna()
min_len = min(len(ofi_lag), len(price_chg))
from scipy import stats
corr, p = stats.pearsonr(ofi_lag.iloc[:min_len], price_chg.iloc[:min_len])
print(f'OFI(t-1) vs 价格变化(t) 相关系数: {corr:.4f} (p={p:.4f})')
print('正相关验证：买压（正OFI）确实对短期价格有向上预测力')


## 🎯 练习

1. 用 OFI 滞后 1 步预测下一步价格变化，计算 Spearman IC，OFI 对短期价格有多少预测力？
2. 模拟「大单隐蔽拆分」：一个 50000 股的大买单被拆成 50 个 1000 股，分析拆单过程中 OFI 积累情况和价格推动。
3. 研究 Kyle's Lambda（价格冲击系数）：用 OLS 回归 价格变化 = Lambda * signed_volume，Lambda 越大意味着该股票流动性越差。

---
**下一节** → `02_market_making.ipynb`